# Ionization balance

Without prescribed drivers, a *closed* steady state needs the total density of each element; with `electron_density=None` the electron density follows from quasi-neutrality, which makes the problem nonlinear.

In [ ]:
import numpy as np
import plasma_plots
import xarray as xr

import opencr

model = opencr.load_model("hydrogen/atomic/basic", n_max=16)
balance = model.solve_steady_state(
    opencr.Conditions(2.0, None, totals={"H": 1e19}), mode="full"
)
print(balance.report.summary())
print(
    f"n_e = {balance.electron_density:.3e} m^-3, ionized fraction {balance.population('H+') / 1e19:.3f}"
)

The ionized fraction against temperature. It is set by $S_\mathrm{CR}/\alpha_\mathrm{CR}$, so it also depends on the density:

In [ ]:
temperatures = np.geomspace(0.5, 10.0, 20)
curves = []
for total in (1e17, 1e19, 1e21):
    data = opencr.scan(
        model,
        opencr.Conditions(1.0, None, totals={"H": total}),
        mode="full",
        lines=False,
        electron_temperature=temperatures,
    )
    curves.append(
        (f"n_H = {total:.0e} m^-3", data["population"].sel(state="H+") / total)
    )
with plasma_plots.figure(1, 1) as fig:
    for label, curve in curves:
        curve.plasma.plot.lineout(ax=fig[0], title="ionized fraction")
        fig[0].lines[-1].set_label(label)
    fig[0].set_xscale("log")
    fig[0].set_ylabel("n(H+) / n_H")
    fig[0].legend()

## Saving a result

`export` writes the populations, rates, fluxes and lines to NetCDF, with the conditions, the solver report and the model's manifest as attributes.

In [ ]:
import tempfile
from pathlib import Path

path = balance.export(Path(tempfile.mkdtemp()) / "balance.nc")
saved = xr.open_dataset(path, engine="scipy")
print(saved["population"].sel(state=["H(n=1)", "H(n=2)", "H+"]).values)
print(saved.attrs["model"], saved.attrs["solve"])